## 02a Experiment 1a: Category Membership - Attributes Task

### 0 Setup

In [6]:
import pandas as pd
import csv
from google import genai
from google.genai import types
import time
import random
import ast

In [7]:
# Category
category = "Clothing"

### 1 Functions

In [8]:
# Prompt Model to answer Questionnaire

def collect_output(client, prompt):
    response = client.models.generate_content(
        model="gemini-3-pro-preview",
        contents=prompt)
    return response

### B. Attributes Task

In [9]:
# CATEGORIES & ATTRIBUTES #from rosch1975c
df = pd.read_csv('01_catmemexp_attributestask_items4attributes.csv')
items_df = df[df['category'] == f"{category}"]
items_df.head()

,model,category,frequency,items
42,Meta-Llama-3.1-70B-Instruct,Clothing,most_frequent,"['jean', 'shoe', 'boot', 'shirt', 'sweater', '..."
43,Meta-Llama-3.1-70B-Instruct,Clothing,average_frequency,"['t-shirt', 'hoodie', 'sandal', 'short', 'unde..."
44,Meta-Llama-3.1-70B-Instruct,Clothing,least_frequent,"['pajama', 'leggings', 'tight', 'overall', 'sw..."
45,Meta-Llama-3.1-8B-Instruct,Clothing,most_frequent,"['shoe', 'boot', 'shirt', 'jacket', 'coat', 'd..."
46,Meta-Llama-3.1-8B-Instruct,Clothing,average_frequency,"['jean', 't-shirt', 'sandal', 'short', 'underw..."


In [ ]:
# Collect Answers from Model

items_reps = 5 #30s

# retrieve API 
with open('', 'r') as file:
    api = file.read()     
client = genai.Client(api_key=api)

prompt_outputs = pd.DataFrame(columns=['model', 'prompt', 'prompt_id', 'runtime', 'category','frequency', 'item','output'])
model_output = pd.DataFrame()

for rep in range(items_reps): # max rep == 30
    
    items_df = items_df[items_df['model'] == f"gemini-3-pro"]
    items_list = items_df['items'].tolist()
    items_list = [ast.literal_eval(item) for item in items_list if isinstance(item, str)]
    items_list = [item for sublist in items_list for item in sublist]

    frequency_list_tmp = items_df['frequency'].tolist()
    frequency_list = [freq for freq in frequency_list_tmp for _ in range(6)]
    paired_items = list(zip(items_list, frequency_list))

    random.shuffle(paired_items)
    shuffled_items_list, shuffled_range_list = zip(*paired_items)
    shuffled_items_list = list(shuffled_items_list)
    shuffled_range_list = list(shuffled_range_list)
        
    for i in range(len(shuffled_items_list)):

            current_prompt = f"""
            Write all of the attributes of the object "{shuffled_items_list[i]}" that you can think of. Write the attributes or characteristics you think are characteristic of the object "{shuffled_items_list[i]}".
            Do not add any further details, only write the items that answer the instruction.
            Provide your answer in plain text as a comma-separated list of adjectives.
            """
            
            start_time = time.time() 
            output = collect_output(client, current_prompt)
            end_time = time.time() 
            runtime = end_time - start_time
            model_output = pd.DataFrame({
                'model': 'gemini-3-pro',
                'prompt': [current_prompt],
                'prompt_id': [6],
                'runtime': [runtime],
                'category': [category],
                'frequency': [shuffled_range_list[i]],
                'item': [shuffled_items_list[i]],
                'output': [output]})
            
            prompt_outputs = pd.concat([prompt_outputs, model_output], ignore_index=True)

prompt_outputs.head()

C:\Users\AS\AppData\Local\Temp\ipykernel_9128\1014386595.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prompt_outputs = pd.concat([prompt_outputs, model_output], ignore_index=True)


,model,prompt,prompt_id,runtime,category,frequency,item,output
0,gemini-3-pro,\n Write all of the attributes of t...,6,12.944468,Clothing,most_frequent,sandal,sdk_http_response=HttpResponse(\n headers=<di...
1,gemini-3-pro,\n Write all of the attributes of t...,6,11.418088,Clothing,least_frequent,shrug,sdk_http_response=HttpResponse(\n headers=<di...
2,gemini-3-pro,\n Write all of the attributes of t...,6,9.113640,Clothing,average_frequency,hijab,sdk_http_response=HttpResponse(\n headers=<di...
3,gemini-3-pro,\n Write all of the attributes of t...,6,8.639854,Clothing,average_frequency,visor,sdk_http_response=HttpResponse(\n headers=<di...
4,gemini-3-pro,\n Write all of the attributes of t...,6,10.814239,Clothing,average_frequency,long john,sdk_http_response=HttpResponse(\n headers=<di...


In [11]:
len(prompt_outputs)

90

In [12]:
prompt_outputs.head()

,model,prompt,prompt_id,runtime,category,frequency,item,output
0,gemini-3-pro,\n Write all of the attributes of t...,6,12.944468,Clothing,most_frequent,sandal,sdk_http_response=HttpResponse(\n headers=<di...
1,gemini-3-pro,\n Write all of the attributes of t...,6,11.418088,Clothing,least_frequent,shrug,sdk_http_response=HttpResponse(\n headers=<di...
2,gemini-3-pro,\n Write all of the attributes of t...,6,9.113640,Clothing,average_frequency,hijab,sdk_http_response=HttpResponse(\n headers=<di...
3,gemini-3-pro,\n Write all of the attributes of t...,6,8.639854,Clothing,average_frequency,visor,sdk_http_response=HttpResponse(\n headers=<di...
4,gemini-3-pro,\n Write all of the attributes of t...,6,10.814239,Clothing,average_frequency,long john,sdk_http_response=HttpResponse(\n headers=<di...


In [13]:
len(prompt_outputs)

90

In [14]:
# Cleaning Model Output

text = ""
response = prompt_outputs['output'][0]

def extract_text(row):
    text = ""
    for candidate in row['output'].candidates:
        for part in candidate.content.parts:
            text += part.text + "\n"  
    return text.strip()  

prompt_outputs['output'] = prompt_outputs.apply(extract_text, axis=1)

In [15]:
prompt_outputs.head()

,model,prompt,prompt_id,runtime,category,frequency,item,output
0,gemini-3-pro,\n Write all of the attributes of t...,6,12.944468,Clothing,most_frequent,sandal,"open, open-toed, strappy, breathable, lightwei..."
1,gemini-3-pro,\n Write all of the attributes of t...,6,11.418088,Clothing,least_frequent,shrug,"cropped, short, knitted, open, lightweight, we..."
2,gemini-3-pro,\n Write all of the attributes of t...,6,9.113640,Clothing,average_frequency,hijab,"modest, religious, Islamic, traditional, cultu..."
3,gemini-3-pro,\n Write all of the attributes of t...,6,8.639854,Clothing,average_frequency,visor,"curved, protective, transparent, opaque, tinte..."
4,gemini-3-pro,\n Write all of the attributes of t...,6,10.814239,Clothing,average_frequency,long john,"long, rectangular, sweet, doughy, soft, froste..."


In [16]:
len(prompt_outputs)

90

In [17]:
# take away NaN Answers
print("Number of rows:", len(prompt_outputs),"\n")
print("Number of rows after NA filtering:", len(prompt_outputs.dropna()))

Number of rows: 90 

Number of rows after NA filtering: 90


In [18]:
#prompt_outputs.to_csv(f'01_catmemexp_attributestask_allmodels_raw_2.csv',index=False, quoting=csv.QUOTE_ALL, encoding='utf-8', mode='a')
prompt_outputs.to_csv(f'gemini_clothing_10newincomplete.csv',index=False, quoting=csv.QUOTE_ALL, encoding='utf-8', mode='a')